In [ ]:
import pandas as pd

# 读取CSV文件
df = pd.read_csv('./data/data_div_train_and_test/train_data.csv', header=None, names=['user', 'item', 'label'])

# 保留标签为1.0的行
positive_examples = df[df['label'] == 1.0]

# 将结果保存到新的CSV文件
positive_examples.to_csv('./data/BPR_div_train_and_test/train_data.csv', sep='\t', header=False, index=False)

In [ ]:

import csv
from collections import defaultdict

# 读取原始数据
raw_data = []
with open('./data/data_div_train_and_test/test_data.csv', 'r', encoding='utf-8') as infile:
    reader = csv.reader(infile)
    for row in reader:
        raw_data.append(row)

# 将数据转换为字典，以用户为键，正例和负例分别为值
user_data = defaultdict(lambda: {'positive': None, 'negative': []})
for row in raw_data:
    user_id, item_id, rating = map(int, map(float, row))
    if rating == 1:
        user_data[user_id]['positive'] = item_id
    else:
        user_data[user_id]['negative'].append(item_id)

# 将数据写入新的 CSV 文件
with open('./data/BPR_div_train_and_test/test_data.csv', 'w', newline='', encoding='utf-8') as outfile:
    for user_id, data in user_data.items():
        positive_item = data['positive']
        negative_items = data['negative']
        line = f"({user_id},{positive_item})"
        line += '\t' + '\t'.join(map(str, negative_items))
        outfile.write(line + '\n')



上面是处理数据集将ml数据集处理成符合代码要求的数据

下面BPR原代码需要的数据和NCF、DMF所需要的数据不一样

BPR的子数据集可以根据原来分好的shard模仿已经有的分好的数据集。

In [ ]:
import pandas as pd
n=3
for part in range(1,n+1):
    # 读取CSV文件
    df = pd.read_csv(f'./data/div_shard{n}/sub_train{part}.csv', header=None, names=['user', 'item', 'label'])

    # 保留标签为1.0的行
    positive_examples = df[df['label'] == 1.0]

    # 将结果保存到新的CSV文件
    positive_examples.to_csv(f'./data/BPR_div_shard{n}/sub_train{part}.csv', sep='\t', header=False, index=False)

In [ ]:
import csv
from collections import defaultdict

n=3
for part in range(1,n+1):
    # 读取原始数据
    raw_data = []
    with open(f'./data/div_shard{n}/sub_test{part}.csv', 'r', encoding='utf-8') as infile:
        reader = csv.reader(infile)
        for row in reader:
            raw_data.append(row)

    # 将数据转换为字典，以用户为键，正例和负例分别为值
    user_data = defaultdict(lambda: {'positive': None, 'negative': []})
    for row in raw_data:
        user_id, item_id, rating = map(int, map(float, row))
        if rating == 1:
            user_data[user_id]['positive'] = item_id
        else:
            user_data[user_id]['negative'].append(item_id)

    # 将数据写入新的 CSV 文件
    with open(f'./data/BPR_div_shard{n}/sub_test{part}.csv', 'w', newline='', encoding='utf-8') as outfile:
        for user_id, data in user_data.items():
            positive_item = data['positive']
            negative_items = data['negative']
            line = f"({user_id},{positive_item})"
            line += '\t' + '\t'.join(map(str, negative_items))
            outfile.write(line + '\n')



以上作废
先处理测试集（带负例），将测试集转化为好处理的格式
在根据用户名处理测试集和训练集，分成子数据集

In [ ]:
import pandas as pd

# 读取原始CSV文件
df = pd.read_csv('./data/BPR_div_train_and_test/test_data.csv', sep='\t', header=None)

# 新的格式
new_rows = []

# 处理每一行
with open('your_new_file.csv', 'w') as f:
    for _, row in df.iterrows():
        user_id, positive_item = eval(row[0])
        new_rows.append([user_id, positive_item, 1])

        for item in row[1:]:
            new_rows.append([user_id, item, 0])

# 转换为DataFrame
new_df = pd.DataFrame(new_rows, columns=['user_id', 'item_id', 'interaction'])

# 保存为新的CSV文件
# sourcce用：new_df.to_csv('./data/BPR_div_train_and_test/test_data_negative.csv', header=False,index=False)
new_df.to_csv('./data/BPR_div_train_and_test/test_data_negative_shard.csv', header=False,index=False)

In [ ]:
import pandas as pd
import numpy as np
import random

# 读取原始CSV文件
data = pd.read_csv(r'./data/BPR_div_train_and_test/train_dataset_negative.csv', sep='\t', names=['user_id', 'item_i', 'item_j'], engine='python')

# 设置随机数种子以确保可复现性
random.seed(41)

# 获取所有不同的user_id
unique_user_ids = data['user_id'].unique()

# 随机打乱user_id的顺序
random.shuffle(unique_user_ids)

# 设置要分成的子数据集数量
num_subsets = 15  # 将划分好的大训练集集分为n份

# 将user_id均匀地分成n份
num_users = len(unique_user_ids)
subset_size = num_users // num_subsets
subsets = []

for i in range(num_subsets):
    if i < num_subsets - 1:
        user_subset = unique_user_ids[i * subset_size: (i + 1) * subset_size]
    else:
        user_subset = unique_user_ids[i * subset_size:]
    
    subset = data[data['user_id'].isin(user_subset)]
    subsets.append(subset)

# 保存n个子数据集为CSV文件
for i, subset in enumerate(subsets):
    subset.to_csv(f'./data/BPR_div_shard{num_subsets}/sub_train{i+1}.csv', sep=',', header=False, index=False)  # 保存到文件subset_1.csv, subset_2.csv, 等等

print(f"训练集数据已按要求划分为{num_subsets}个子数据集并保存为CSV文件。")

# 现在处理测试集
test_data = pd.read_csv(r'./data/BPR_div_train_and_test/test_data_nagetive.csv', sep='\t',names=['user_id', 'item_i', 'item_j'], engine='python')
test_subsets = []

for i, user_subset in enumerate(subsets):
    test_subset = test_data[test_data['user_id'].isin(user_subset['user_id'].unique())]
    test_subsets.append(test_subset)

# 保存n个子测试集为CSV文件
for i, test_subset in enumerate(test_subsets):
    test_subset.to_csv(f'./data/BPR_div_shard{num_subsets}/sub_test{i+1}.csv', sep=',', header=False, index=False)

print(f"测试集数据已按要求划分为{num_subsets}个子数据集并保存为CSV文件。")

训练集数据已按要求划分为15个子数据集并保存为CSV文件。
测试集数据已按要求划分为15个子数据集并保存为CSV文件。
